In [46]:
import pandas as pd
import numpy as np
import pickle
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder, MultiLabelBinarizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

In [47]:
# Load datasets
sym_df = pd.read_csv("DiseaseAndSymptoms.csv")
pre_df = pd.read_csv("Disease precaution.csv")
sym_df['Disease'] = sym_df['Disease'].str.strip()
pre_df['Disease'] = pre_df['Disease'].str.strip()

In [48]:
# Combine symptom columns
symptom_cols = [c for c in sym_df.columns if c.startswith('Symptom')]

def get_symptoms(row):
    symptoms = []
    for col in symptom_cols:
        val = row[col]
        if pd.notna(val):
            symptoms.append(str(val).strip().replace(' ', '_').lower())
    return symptoms

sym_df['symptom_list'] = sym_df.apply(get_symptoms, axis=1)

In [49]:
# Deduplicate
sym_df['symptom_str'] = sym_df['symptom_list'].apply(lambda x: ','.join(sorted(x)))
sym_df = sym_df.drop_duplicates(subset=['Disease', 'symptom_str'])
print(f"Unique rows after dedup: {len(sym_df)}")

Unique rows after dedup: 304


In [50]:
# Get all unique symptoms
all_symptoms = set()
for s_list in sym_df['symptom_list']:
    all_symptoms.update(s_list)
all_symptoms = sorted(list(all_symptoms))
print(f"Total unique symptoms: {len(all_symptoms)}")

Total unique symptoms: 131


In [51]:
# Augment
np.random.seed(42)
augmented_rows = []
for _, row in sym_df.iterrows():
    disease = row['Disease']
    symptoms = row['symptom_list']
    augmented_rows.append({'Disease': disease, 'symptom_list': symptoms})
    if len(symptoms) >= 3:
        for _ in range(4):
            n_drop = np.random.randint(1, min(3, len(symptoms)))
            partial = list(np.random.choice(symptoms,
                          size=len(symptoms) - n_drop,
                          replace=False))
            augmented_rows.append({'Disease': disease, 'symptom_list': partial})

aug_df = pd.DataFrame(augmented_rows)
print(f"Augmented dataset size: {len(aug_df)}")

Augmented dataset size: 1520


In [53]:
mlb = MultiLabelBinarizer(classes=all_symptoms)
X = mlb.fit_transform(aug_df['symptom_list'])        # ← aug_df
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(aug_df['Disease'])   # ← aug_df

print(f"Feature matrix shape: {X.shape}")

Feature matrix shape: (1520, 131)


In [54]:
# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Training: {X_train.shape[0]}, Testing: {X_test.shape[0]}")

Training: 1216, Testing: 304


In [55]:
# Train
model = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')
model.fit(X_train, y_train)
print("Model trained!")

Model trained!


In [56]:
# Evaluate
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f"\n✅ Accuracy: {accuracy * 100:.2f}%")
print(classification_report(y_test, y_pred,
      target_names=label_encoder.classes_, zero_division=0))


✅ Accuracy: 97.70%
                                         precision    recall  f1-score   support

(vertigo) Paroymsal  Positional Vertigo       0.88      1.00      0.93         7
                                   AIDS       1.00      1.00      1.00         5
                                   Acne       1.00      1.00      1.00         5
                    Alcoholic hepatitis       1.00      1.00      1.00         8
                                Allergy       1.00      1.00      1.00         5
                              Arthritis       1.00      1.00      1.00         6
                       Bronchial Asthma       1.00      1.00      1.00         7
                   Cervical spondylosis       1.00      1.00      1.00         6
                            Chicken pox       1.00      1.00      1.00        10
                    Chronic cholestasis       1.00      0.88      0.93         8
                            Common Cold       1.00      1.00      1.00         9
       

In [44]:
# Evaluate
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f"\n✅ Accuracy: {accuracy * 100:.2f}%")
print(classification_report(y_test, y_pred,
      target_names=label_encoder.classes_, zero_division=0))


✅ Accuracy: 100.00%
                                         precision    recall  f1-score   support

(vertigo) Paroymsal  Positional Vertigo       1.00      1.00      1.00         1
                                   AIDS       1.00      1.00      1.00         1
                                   Acne       1.00      1.00      1.00         1
                    Alcoholic hepatitis       1.00      1.00      1.00         2
                                Allergy       1.00      1.00      1.00         1
                              Arthritis       1.00      1.00      1.00         1
                       Bronchial Asthma       1.00      1.00      1.00         1
                   Cervical spondylosis       1.00      1.00      1.00         1
                            Chicken pox       1.00      1.00      1.00         2
                    Chronic cholestasis       1.00      1.00      1.00         2
                            Common Cold       1.00      1.00      1.00         2
      

In [57]:
# Save
pickle.dump(model, open("disease_model.pkl", "wb"))
pickle.dump(mlb, open("mlb.pkl", "wb"))
pickle.dump(label_encoder, open("label_encoder.pkl", "wb"))
print("\n✅ All models saved!")


✅ All models saved!
